<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 13 — Learning from Examples

Every program so far followed rules that you wrote. This session is about
programs that are given examples and find a rule themselves: **machine
learning**. Getting such a program to produce predictions takes three lines.
The work, and this session, is in the question that comes after: *are the
predictions any good, and how would you know?*

Six ideas, one part each:

| Part | Idea | In one sentence |
|---|---|---|
| 1 | Features and target | A model learns to predict one column from the others. |
| 2 | Train and test | A model is measured on examples it has not seen, or not at all. |
| 3 | The baseline | Before any model: how good is the simplest possible guess? |
| 4 | A decision tree | A model that asks yes/no questions, chosen from the training examples. |
| 5 | Overfitting | A model can learn its examples by heart and get worse at everything else. |
| 6 | Beyond accuracy | Which mistakes it makes matters as much as how many. |

**How this session's notebooks fit together**

| Notebook | What it is |
|---|---|
| `01_Learning_from_Examples` (this one) | The ideas, on data you can draw |
| `02_Late_Answers_Example` | One real problem from start to finish: a model that is 97.5 % accurate and useless |
| `03_Machine_Learning_Reference` | A lookup guide to keep open while you do Homework 12 |

**What is not repeated here.** Tables, missing values and `groupby` are
Session 8; plots are Session 9; seeds and "an estimate changes from run to
run" are Session 10. All of them are used below.

**Before you start**

- Set the kernel (upper right corner) to **Conda Python 3.12**.
- Run the cells in order with **Shift + Enter**. Before each cell, guess what
  it will show.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

pd.options.display.float_format = "{:,.3f}".format

---

## Part 1 — Features and target

The coffee cart of Sessions 9 and 10 wants to know in the morning whether the
day will be **busy**, so that it can bring a second person. It has a year of
records: the temperature at noon, whether it rained, whether it was a
weekend, and whether the day turned out busy. (Made-up data again.)

In [ ]:
rng = np.random.default_rng(13)
DAYS = 365

days = pd.DataFrame({
    "temperature": np.round(rng.normal(16, 8, DAYS), 1),
    "rain": (rng.random(DAYS) < 0.3).astype(int),
    "weekend": (np.arange(DAYS) % 7 >= 5).astype(int),
})
# Cold, dry weekend days are busy -- with a good deal of chance on top.
pull = 0.8 - 0.22 * (days["temperature"] - 16) + 1.6 * days["weekend"] - 1.4 * days["rain"]
days["busy"] = (rng.random(DAYS) < 1 / (1 + np.exp(-pull + 1.2))).astype(int)

print(days.shape, " busy days: {:.0%}".format(days["busy"].mean()))
days.head(8)

You could write the rule yourself: "busy if it is a weekend and below 15
degrees", say. How would you pick 15? You would look at the records. Machine
learning is that step, done by a program: **the records are the input, and the
rule is the output.**

For that, the table is cut into two pieces.

| | Name | Here | Shape |
|---|---|---|---|
| what the model may look at | **features**, `X` | temperature, rain, weekend | a table: one row per day, one column per feature |
| what it should predict | **target**, `y` | busy | one value per day |

When the target is a category (busy or not), the task is called
**classification**. When it is a number (how many cups), it is **regression**.
This session does classification with two classes, 0 and 1.

In [ ]:
X = days[["temperature", "rain", "weekend"]]
y = days["busy"]
print(X.shape, y.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
jitter = rng.uniform(-0.15, 0.15, DAYS)
for busy, colour, marker, name in [(0, "lightsteelblue", "o", "quiet"), (1, "firebrick", "^", "busy")]:
    rows = days[days["busy"] == busy]
    ax.scatter(rows["temperature"], rows["weekend"] + jitter[rows.index], color=colour,
               marker=marker, alpha=0.7, label=name)
ax.set_yticks([0, 1])
ax.set_yticklabels(["weekday", "weekend"])
ax.set_xlabel("temperature at noon (°C)")
ax.set_ylabel("")
ax.set_title("Busy days are mostly cold, and more often weekends -- not always")
ax.legend()
fig.tight_layout()

The red triangles lean left and up, and there is no clean border. Some warm
weekdays were busy; some cold weekends were quiet. No rule will get every day
right, because the features do not contain everything that makes a day busy.
That is the normal situation, and it is why the result of all this will be a
*percentage*, not a proof.

---

## Part 2 — Train and test

Here is a model, trained and scored in three lines. Do not worry yet about
what a tree is.

In [ ]:
model = DecisionTreeClassifier(random_state=13)
model.fit(X, y)                          # learn from all 365 days
print("accuracy:", model.score(X, y))    # share of the days it gets right

Nearly perfect. Too good, given the picture above, in which busy and quiet
days were thoroughly mixed. What happened?

The model was scored on the very days it learned from. That is an exam whose
questions were handed out the week before: a high mark shows that the answers
were remembered, not that anything was understood. This model has remembered
365 days.

The cure is the most important habit in the whole subject. **Before any
learning, put some rows aside.** The model learns from the rest, the
**training set**, and is measured on the rows it has never seen, the **test
set**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=13)

print(len(X_train), "days to learn from,", len(X_test), "put aside")

model = DecisionTreeClassifier(random_state=13)
model.fit(X_train, y_train)

print("accuracy on the days it learned from: {:.3f}".format(model.score(X_train, y_train)))
print("accuracy on days it has not seen:     {:.3f}".format(model.score(X_test, y_test)))

The second number is the honest one, and it is a long way below the first.
Only the test accuracy says anything about tomorrow, because tomorrow is a day
the model has not seen.

`train_test_split` shuffles the rows and deals them into two parts, keeping
each day's features and its answer together. `random_state` is the seed
(Session 10): the same seed gives the same split. Homework 12 has you write
this function yourself.

Three rules follow, and breaking any of them gives a number that looks better
than it is.

1. The test set is put aside **first**, before looking at anything.
2. Nothing that is learned or chosen may use it.
3. It is used **once**, at the end.

**✏️ Try it:** Change `random_state=13` in the split to 14, then 15. How much does the test accuracy move? What does Session 10 say about a number computed from 92 days?

---

## Part 3 — The baseline

Is the test accuracy above good? Compared with what?

The simplest "model" does not look at the features at all. It answers with
whatever is more common in the training set, every time.

In [ ]:
print(y_train.value_counts())

majority = y_train.value_counts().idxmax()
baseline = (y_test == majority).mean()
print()
print("always answering {}: right for {:.3f} of the test days".format(majority, baseline))

That is the **baseline**, and it changes how every other number reads. An
accuracy of 65 % sounds like something until you learn that saying "quiet"
every day already gets 57 %, with no data and no computer. The model has
earned the difference, not the whole.

The baseline is not a formality. On data where one class is rare it can be
very high: if 2 % of transactions are fraud, "never fraud" is 98 % accurate
and catches nothing. Notebook 02 is about exactly that.

Note that the majority was read off the **training** set and then checked on
the test set. Even this guess is "learned", and learning uses training rows
only.

---

## Part 4 — A decision tree

A **decision tree** predicts by asking yes/no questions about the features,
one after another, like the game of twenty questions. Each question compares
one feature with a number. Training means *choosing the questions*.

Start with a tree that may ask one question. Which question is best? Try them
all. For the temperature: every possible threshold, and for each, predict the
majority on each side and count how many training days come out right.

In [ ]:
def accuracy_of_question(feature, threshold):
    """Accuracy on the training set of one question: is feature <= threshold?"""
    below = X_train[feature] <= threshold
    right = 0
    for side in (below, ~below):
        if side.sum() > 0:
            answer = y_train[side].value_counts().idxmax()     # the majority on this side
            right += (y_train[side] == answer).sum()
    return right / len(y_train)


thresholds = np.arange(0, 32, 1.0)
by_threshold = [accuracy_of_question("temperature", t) for t in thresholds]
best = thresholds[int(np.argmax(by_threshold))]

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(thresholds, by_threshold, marker="o", color="steelblue")
ax.axvline(best, color="firebrick", linestyle="--", label="best: {:.0f} °C".format(best))
ax.set_xlabel("threshold for 'temperature <= ...?' (°C)")
ax.set_ylabel("training days predicted right")
ax.set_title("One question, every possible threshold")
ax.legend()

print("weekend <= 0.5 ?  {:.3f}".format(accuracy_of_question("weekend", 0.5)))
print("rain <= 0.5 ?     {:.3f}".format(accuracy_of_question("rain", 0.5)))
print("temperature <= {:.0f} ?  {:.3f}".format(best, max(by_threshold)))
fig.tight_layout()

That search is all "learning" is, for a tree: try the candidate questions,
keep the one that sorts the training examples best. Then do the same again on
each side, with the examples that landed there, and so on down.

scikit-learn does the search for you (with a slightly cleverer measure than
accuracy, called Gini impurity; the idea is the same).

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=13)
tree.fit(X_train, y_train)

print(export_text(tree, feature_names=list(X.columns)))
print("test accuracy: {:.3f}   (baseline {:.3f})".format(tree.score(X_test, y_test), baseline))

Read it from the top. Each indented line is a question; `class: 1` at the end
of a branch is the verdict "busy". Two questions deep, four verdicts. (Here the
second question changes no verdict: both branches on the cold side say busy.
The tree asked it because it sorts the training days a little more cleanly,
not because the answer flips.) You can check a small tree against your own
sense of a coffee cart, which is its great advantage: **it can be read.**

The whole of scikit-learn follows the pattern you just saw:

| Step | Code | Means |
|---|---|---|
| make | `model = DecisionTreeClassifier(max_depth=2, random_state=13)` | choose the kind of model and its settings |
| learn | `model.fit(X_train, y_train)` | find the rule from the training examples |
| predict | `model.predict(X_test)` | apply the rule: one answer per row |
| score | `model.score(X_test, y_test)` | accuracy on those rows |

A tree with two features can also be drawn. Each question cuts the plane with
a straight line, so the verdicts are rectangles:

In [ ]:
def draw_regions(ax, model, title):
    """Shade where a model trained on temperature and weekend predicts 'busy'."""
    temperatures = np.linspace(-8, 40, 300)
    for weekend in (0, 1):
        grid = pd.DataFrame({"temperature": temperatures, "weekend": weekend})
        verdict = model.predict(grid)
        ax.fill_between(temperatures, weekend - 0.4, weekend + 0.4, where=verdict == 1,
                        color="mistyrose", step="mid")
    shake = np.random.default_rng(0).uniform(-0.3, 0.3, len(X_train))
    for busy, colour, marker in [(0, "steelblue", "o"), (1, "firebrick", "^")]:
        rows = (y_train == busy).to_numpy()
        ax.scatter(X_train["temperature"][rows], X_train["weekend"][rows] + shake[rows],
                   color=colour, marker=marker, s=14, alpha=0.7)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["weekday", "weekend"])
    ax.set_xlabel("temperature (°C)")
    ax.set_title(title)
    return ax


two_features = ["temperature", "weekend"]
small_tree = DecisionTreeClassifier(max_depth=2, random_state=13).fit(X_train[two_features], y_train)

fig, ax = plt.subplots(figsize=(8, 3.2))
draw_regions(ax, small_tree, "A tree of depth 2: where it says 'busy' (shaded)")
fig.tight_layout()

**✏️ Try it:** Train the tree with `max_depth=1` and print it. Is its one question the one the threshold search above found?

---

## Part 5 — Overfitting

`max_depth` limits how many questions the tree may ask in a row. What if it
may ask as many as it likes?

In [ ]:
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=13).fit(X_train[two_features], y_train)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2), sharey=True)
draw_regions(axes[0], small_tree, "Depth 2: {} verdicts".format(small_tree.get_n_leaves()))
draw_regions(axes[1], deep_tree, "No limit: {} verdicts".format(deep_tree.get_n_leaves()))
fig.tight_layout()

On the left, one broad region: cold days are busy. On the right, a
bar code. The unlimited tree has cut the temperature axis into slivers, each
there to get one or two particular training days right. "Busy between 11.3 and
11.6 degrees on weekdays, quiet between 11.6 and 12.1" is not knowledge about
coffee. It is a description of last year's accidents.

That is **overfitting**: the model fits the training examples, noise
included, and so does worse on anything new. The way to see it is to measure
both accuracies while the tree is allowed to grow:

In [ ]:
rows = []
for depth in (1, 2, 3, 4, 6, 8, 12, 20):
    grown = DecisionTreeClassifier(max_depth=depth, random_state=13).fit(X_train, y_train)
    rows.append({"depth": depth, "leaves": grown.get_n_leaves(),
                 "train": grown.score(X_train, y_train), "test": grown.score(X_test, y_test)})
growth = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(growth["depth"], growth["train"], marker="o", label="days it learned from")
ax.plot(growth["depth"], growth["test"], marker="s", linestyle="--", label="days it has not seen")
ax.axhline(baseline, color="gray", linestyle=":", label="baseline")
ax.set_xlabel("depth the tree is allowed")
ax.set_ylabel("accuracy")
ax.set_title("More questions: better on the training days, not on new ones")
ax.legend()
growth
fig.tight_layout()

The training accuracy can only rise: a deeper tree can always fit its own
examples at least as well. The test accuracy rises a little, then stalls or
falls. The widening gap between the two lines is the picture of overfitting.

The remedy is restraint: a shallower tree. But **which depth?** It is
tempting to pick the row with the best test accuracy. Rule 2 of Part 2 forbids
it: that would be using the test set to choose, and the test accuracy of the
chosen tree would no longer be an honest estimate.

The answer is a third set. Split the *training* set again, into a part to fit
on and a **validation set** to choose on. The test set stays untouched until
the very end.

```text
 all rows ──▶ training set ──▶ fit part          trees are trained here
          │                └──▶ validation part  the depth is chosen here
          └──▶ test set                          the final model is measured here, once
```

In [ ]:
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train, y_train, test_size=0.25, random_state=13)

best_depth, best_score = None, -1.0
for depth in (1, 2, 3, 4, 6, 8, 12, 20):
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=13).fit(X_fit, y_fit)
    score = candidate.score(X_valid, y_valid)
    print("depth {:2}: validation accuracy {:.3f}".format(depth, score))
    if score > best_score:                    # strictly better: ties keep the smaller tree
        best_depth, best_score = depth, score

final = DecisionTreeClassifier(max_depth=best_depth, random_state=13).fit(X_train, y_train)
print()
print("chosen depth: {}   test accuracy, measured once: {:.3f}   baseline: {:.3f}".format(
    best_depth, final.score(X_test, y_test), baseline))

**✏️ Try it:** Rerun the last two cells with another `random_state` in the first split of Part 2. Is the same depth chosen? Does the final tree still beat the baseline, and by how much?

---

## Part 6 — Beyond accuracy

Accuracy counts mistakes. It does not say **which** mistakes, and for the cart
the two kinds are not alike. Predicting "quiet" on a day that turns out busy
means a queue down the street. Predicting "busy" on a quiet day means paying a
second person to stand around.

Lay the predictions out against what happened. With "busy" as the thing we are
looking for:

In [ ]:
predicted = final.predict(X_test)
actual = y_test.to_numpy()

tp = int(((predicted == 1) & (actual == 1)).sum())      # said busy, was busy
fp = int(((predicted == 1) & (actual == 0)).sum())      # said busy, was quiet: a false alarm
fn = int(((predicted == 0) & (actual == 1)).sum())      # said quiet, was busy: a miss
tn = int(((predicted == 0) & (actual == 0)).sum())      # said quiet, was quiet

pd.DataFrame({"was busy": [tp, fn], "was quiet": [fp, tn]},
             index=["predicted busy", "predicted quiet"])

This table is the **confusion matrix**. Its four cells have names:

| | Was busy | Was quiet |
|---|---|---|
| **Predicted busy** | true positive (tp) | false positive (fp): a false alarm |
| **Predicted quiet** | false negative (fn): a miss | true negative (tn) |

Three scores come out of it:

In [ ]:
print("accuracy   {:.3f}   how often is it right?                        (tp + tn) / all".format((tp + tn) / len(actual)))
print("precision  {:.3f}   when it says busy, how often is it busy?      tp / (tp + fp)".format(tp / (tp + fp)))
print("recall     {:.3f}   of the busy days, how many does it announce?  tp / (tp + fn)".format(tp / (tp + fn)))

**Precision** is about false alarms and **recall** is about misses. A cart
that hates queues wants high recall and will accept some wasted shifts. One
with a tight budget wants high precision.

And the baseline, which always says the majority?

In [ ]:
always = np.full(len(actual), majority)
print("it predicts 'busy' on", int((always == 1).sum()), "days")
print("recall:", ((always == 1) & (actual == 1)).sum() / (actual == 1).sum())

It never announces a busy day. Recall 0. Its precision is 0 divided by 0: of
the days it called busy, of which there are none, how many were? That has no
answer, and is reported as "not a number", not as 0.

So a model can lose to the baseline on accuracy and still be the useful one,
or match it on accuracy and be useless. One number is not enough to describe a
classifier. Report the baseline, the accuracy, and the two scores that say
which mistakes are being made.

### What a model is not

A trained model looks like knowledge. Three cautions before you trust one.

**It finds what goes together, not what causes what.** The tree says cold
days are busy. That is a pattern in last year's records. It would say the same
if cold days happened to fall in exam season and it was the exams that sold
the coffee.

**It knows only the world it was shown.** A model trained on last year
predicts next year only as far as next year resembles it. A new café next door
is not in the data.

**It will use any column that helps, including ones it should not have.**
Suppose the table had a column "second person was called in". It predicts
"busy" almost perfectly, because it was written down *after* the day was
busy. A feature that would not be known at the moment of prediction is called
**leakage**, and its symptom is a score that is too good to be true. Notebook
02 walks into one.

**✏️ Try it:** Which would you rather have for a smoke detector: high precision or high recall? And for a spam filter that deletes messages?

---

## Exercises

All on the `days` table.

### 🧩 Exercise 1 — By hand

Without scikit-learn: use the rule "busy if weekend, quiet otherwise" on the
test set. Compute its accuracy, and its confusion matrix, precision and
recall. Does this one-line rule beat the baseline?

### 🧩 Exercise 2 — Split it yourself

Write `my_split(X, y, test_share, seed)` with `np.random.default_rng(seed)`
and `rng.permutation(len(X))`: the first `round(len(X) * test_share)` shuffled
positions are the test rows. Check that every row is in exactly one part and
that `X_test.index` equals `y_test.index`.

### 🧩 Exercise 3 — A feature that leaks

Add a column `second_person` that equals `busy` on nine days out of ten and
is random on the rest (`np.where(rng.random(365) < 0.9, days["busy"], ...)`).
Train a tree of depth 2 with it among the features. What is the test
accuracy? Print the tree. Why would this model be useless on the morning it
is needed?

### 🧩 Exercise 4 — How steady is the result?

Repeat the whole procedure (split, choose the depth on a validation set,
measure once on the test set) for twenty different seeds. Collect the final
test accuracy and the baseline each time. In how many of the twenty does the
tree beat the baseline? Report the mean difference with a standard error, as
in Session 10.

---

## Summary

| Idea | The habit | The tool |
|---|---|---|
| Features and target | Say what is predicted, and from what | `X`, `y` |
| Train and test | Put the test rows aside first; use them once | `train_test_split`, a seed |
| The baseline | Compute it before any model | the majority class |
| A decision tree | Start small; read the tree | `DecisionTreeClassifier`, `export_text` |
| Overfitting | Watch training and test accuracy together | a table over depths; a validation set to choose |
| Beyond accuracy | Report which mistakes, not only how many | the confusion matrix, precision, recall |

### Where each idea appears in Homework 12

| Idea | Homework 12 |
|---|---|
| Features, a target, and keeping the answer out | Phase 1, `build_features` |
| A seeded split | Phase 2, `split` |
| The majority baseline | Phase 3, `majority_class`, `baseline_accuracy` |
| The four counts and the three scores | Phase 4, `confusion`, `scores` |
| A tree at a given depth; the table that shows overfitting | Phase 5, `fit_and_score`, `depth_sweep` |
| Choosing the depth on a validation set | Phase 6, `choose_depth`, `build_report` |

**Next:** `02_Late_Answers_Example` runs into a baseline of 97.5 %, and
`03_Machine_Learning_Reference` is the lookup guide.